## Dataset de ciudades 
Ciudades usadas para promediar clima por país en Open-Meteo: capital + 2 más pobladas (ver D21). Recalculada completa cada vez que se re-corre (no es incremental).

SimpleMaps World Cities Database (versión básica, gratis, CC BY 4.0 — se puede citar y usar en un repo público). Trae city, country, iso3, lat, lng, population, capital por fila — justo lo que necesitamos para derivar capital + 2 ciudades más pobladas por país.

In [0]:
%sql
CREATE OR REPLACE TABLE happiness.meta.config_ciudades_clima
COMMENT 'Ciudades usadas para promediar clima por país en Open-Meteo: capital + 2 más pobladas (ver D21). Recalculada completa cada vez que se re-corre (no es incremental).'
AS
WITH ciudades_tipadas AS (
  -- Casteamos recién acá, no en bronze: esta es una tabla de configuración derivada,
  -- no una tabla bronze, así que no aplica la regla D10 de "todo STRING".
  -- Necesitamos population numérica para rankear y lat/lng numéricos para pasarlos a la API.
  SELECT
    iso3,
    country,
    city,
    city_ascii,
    TRY_CAST(lat AS DOUBLE)        AS lat,
    TRY_CAST(lng AS DOUBLE)        AS lng,
    TRY_CAST(population AS DOUBLE) AS population,
    capital
  FROM read_files(
    '/Volumes/happiness/landing/archivos/SIMPLE MAPS/worldcities.csv',
    format => 'csv',
    inferColumnTypes => false
  )
  WHERE iso3 IS NOT NULL AND iso3 != ''
),
rankeadas AS (
  SELECT
    *,
    -- la capital ('primary') siempre entra primero, pase lo que pase con su población;
    -- los otros 2 lugares se llenan por población descendente
    ROW_NUMBER() OVER (
      PARTITION BY iso3
      ORDER BY
        CASE WHEN capital = 'primary' THEN 0 ELSE 1 END,
        population DESC NULLS LAST
    ) AS rn
  FROM ciudades_tipadas
)
SELECT iso3, country, city, city_ascii, lat, lng, population, capital, rn AS prioridad
FROM rankeadas
WHERE rn <= 3;


## Exportación de lista de ciudades en orden fijo
Se exporta la lista de ciudades en un orden fijo (esto importa: la posición en este CSV va a ser la misma que usemos para reconstruir identidad después en bronze) y descargá el resultado como CSV

In [0]:
%sql
SELECT iso3, country, city, city_ascii, lat, lng, capital
FROM happiness.meta.config_ciudades_clima
ORDER BY iso3, prioridad;

## Descarga de archivos json del clima por ciudad de forma local
Se corre en Python local, no Databricks. Agrupa de a 10 ciudades por llamada (645 ciudades → 13 llamadas en total), mantiene el mismo orden del CSV, y es reanudable (si se corta, lo volvés a correr y saltea los batches que ya bajó):

In [0]:
import requests
import csv
import time
from pathlib import Path

# -- Parámetros (D21: capital + 2 más pobladas, 2011-2025; D22: batching por rate limit) --
ARCHIVO_CIUDADES = "config_ciudades_clima.csv"   # exportado desde happiness.meta.config_ciudades_clima
CARPETA_SALIDA = Path("open_meteo_batches")
FECHA_INICIO = "2011-01-01"
FECHA_FIN = "2025-12-31"
VARIABLES_DIARIAS = "temperature_2m_mean,precipitation_sum,sunshine_duration"
BATCH_SIZE = 10
MAX_REINTENTOS = 3
PAUSA_ENTRE_BATCHES_SEG = 15   # antes 3 — bajamos bastante el ritmo

CARPETA_SALIDA.mkdir(exist_ok=True)

# Leemos la lista en el MISMO orden que quedó en el CSV (= mismo orden que la tabla de config)
with open(ARCHIVO_CIUDADES, newline="", encoding="utf-8") as f:
    ciudades = list(csv.DictReader(f))

print(f"Ciudades totales: {len(ciudades)}")


def llamar_batch(lote):
    lats = ",".join(c["lat"] for c in lote)
    lons = ",".join(c["lng"] for c in lote)
    params = {
        "latitude": lats,
        "longitude": lons,
        "start_date": FECHA_INICIO,
        "end_date": FECHA_FIN,
        "daily": VARIABLES_DIARIAS,
        "timezone": "UTC",
    }
    espera = 10   # antes 5
    for intento in range(1, MAX_REINTENTOS + 1):
        resp = requests.get("https://archive-api.open-meteo.com/v1/archive", params=params, timeout=60)
        if resp.status_code == 429:
            print(f"  429, esperando {espera}s (intento {intento}/{MAX_REINTENTOS})")
            time.sleep(espera)
            espera *= 2
            continue
        if resp.status_code >= 400:
            print(f"  ERROR {resp.status_code}: {resp.text}")   # <-- nueva línea: queremos ver el motivo real
            resp.raise_for_status()
        return resp.text
    return None


num_batches = (len(ciudades) + BATCH_SIZE - 1) // BATCH_SIZE

for i in range(num_batches):
    numero_batch = i + 1
    archivo_salida = CARPETA_SALIDA / f"open_meteo_batch_{numero_batch:02d}.json"

    if archivo_salida.exists():
        print(f"Batch {numero_batch}/{num_batches} ya existe, lo salteo")
        continue

    lote = ciudades[i * BATCH_SIZE : (i + 1) * BATCH_SIZE]
    texto = llamar_batch(lote)

    if texto is None:
        print(f"FALLÓ el batch {numero_batch}/{num_batches} (se agotaron los reintentos)")
        continue

    archivo_salida.write_text(texto, encoding="utf-8")
    print(f"Batch {numero_batch}/{num_batches} OK ({len(lote)} ciudades)")
    time.sleep(PAUSA_ENTRE_BATCHES_SEG)

print("Listo.")

faltantes = [i + 1 for i in range(num_batches)
             if not (CARPETA_SALIDA / f"open_meteo_batch_{i+1:02d}.json").exists()]
print(f"Listo. Faltan {len(faltantes)} de {num_batches} batches: {faltantes}")

## Subida de archivos json a Volume
Se suben los archivos de tipo (open_meteo_batch_01.json, 02, 04, 06, 08) a /Volumes/happiness/landing/archivos/OPEN_METEO/ y luego se hace CREATE OR REPLACE (es Full Load, como todo bronze: reemplaza la tabla entera, no duplica).

In [0]:
%sql
CREATE OR REPLACE TABLE happiness.bronze.open_meteo
COMMENT 'Clima diario por ciudad (capital + 2 más pobladas por país, ver D21) desde la
Archive API de Open-Meteo, 2011-2025. La ingesta real (llamada a la API) se hizo fuera
de Databricks por un script local en lotes de 10 ciudades (ver D22: la API gratuita
se satura por volumen de datos y por límites de tasa en infraestructura compartida).
Cada archivo de landing es un batch con varias ciudades; acá se parsea el array JSON,
se reconstruye la identidad de cada ciudad por posición (no por lat/lon, que la API
redondea a su grilla) y se explotan los días a filas.'
AS
WITH archivos_crudos AS (
  -- cada archivo es UN batch completo devuelto por la API (varias ciudades juntas).
  -- lo leemos como texto plano, una fila por archivo entero (no por línea), porque
  -- la respuesta de Open-Meteo viene compacta, sin saltos de línea internos
  SELECT
    value AS contenido_json,
    _metadata.file_name AS nombre_archivo
  FROM read_files(
    '/Volumes/happiness/landing/archivos/OPEN_METEO/',
    format => 'text'
  )
),
archivos_con_batch AS (
  SELECT
    contenido_json,
    nombre_archivo,
    -- el número de batch sale del nombre de archivo (open_meteo_batch_07.json -> 7);
    -- lo necesitamos para reconstruir la posición global de cada ciudad más abajo
    CAST(regexp_extract(nombre_archivo, 'batch_(\\d+)\\.json', 1) AS INT) AS numero_batch
  FROM archivos_crudos
),
ciudades_explotadas AS (
  SELECT
    nombre_archivo,
    numero_batch,
    pos AS posicion_en_batch,
    ciudad.latitude     AS latitude_devuelta,
    ciudad.longitude    AS longitude_devuelta,
    ciudad.elevation    AS elevation,
    ciudad.location_id  AS location_id,
    ciudad.daily.time                 AS dias,
    ciudad.daily.temperature_2m_mean  AS temperaturas,
    ciudad.daily.precipitation_sum    AS precipitaciones,
    ciudad.daily.sunshine_duration    AS horas_sol
  FROM archivos_con_batch
  -- parseamos el array JSON del batch con schema explícito, todo STRING (D10).
  -- posexplode da la posición 0-indexada de cada ciudad dentro del array del batch:
  -- es la única identidad confiable (location_id falta en la posición 0, y lat/lon
  -- vienen redondeadas a la grilla de la API, no sirven para reconciliar exacto)
  LATERAL VIEW posexplode(
    from_json(
      contenido_json,
      'array<struct<
        latitude: string,
        longitude: string,
        elevation: string,
        location_id: string,
        daily: struct<
          time: array<string>,
          temperature_2m_mean: array<string>,
          precipitation_sum: array<string>,
          sunshine_duration: array<string>
        >
      >>'
    )
  ) AS pos, ciudad
),
ciudades_con_identidad AS (
  SELECT
    e.*,
    -- BATCH_SIZE=10 tiene que coincidir siempre con el del script de descarga
    (e.numero_batch - 1) * 10 + e.posicion_en_batch AS posicion_global
  FROM ciudades_explotadas e
),
config_con_posicion AS (
  -- MISMO orden que se usó para armar los batches en el script (ORDER BY iso3, prioridad)
  SELECT
    iso3, country, city, city_ascii,
    ROW_NUMBER() OVER (ORDER BY iso3, prioridad) - 1 AS posicion_global
  FROM happiness.meta.config_ciudades_clima
),
dias_explotados AS (
  SELECT
    c.iso3, c.country, c.city, c.city_ascii,
    cc.nombre_archivo, cc.latitude_devuelta, cc.longitude_devuelta, cc.elevation, cc.location_id,
    dia.dias           AS fecha,
    dia.temperaturas    AS temperatura_media,
    dia.precipitaciones AS precipitacion_mm,
    dia.horas_sol       AS horas_sol_seg
  FROM ciudades_con_identidad cc
  JOIN config_con_posicion c
    ON c.posicion_global = cc.posicion_global
  -- explotamos los 4 arrays paralelos juntos (arrays_zip), para que cada fila sea
  -- un día completo con sus 3 variables alineadas, no 4 explodes separados
  LATERAL VIEW explode(
    arrays_zip(cc.dias, cc.temperaturas, cc.precipitaciones, cc.horas_sol)
  ) AS dia
)
SELECT
  iso3, country, city, city_ascii,
  latitude_devuelta, longitude_devuelta, elevation, location_id,
  fecha, temperatura_media, precipitacion_mm, horas_sol_seg,
  current_timestamp() AS _ingest_ts,
  nombre_archivo       AS _source
FROM dias_explotados;

In [0]:
%sql
SELECT COUNT(*), COUNT(DISTINCT iso3,city_ascii) from happiness.bronze.open_meteo;

SELECT city_ascii, COUNT(DISTINCT iso3) AS paises
FROM happiness.bronze.open_meteo  -- o sobre el SELECT de la CTE si todavía no la materializaste
GROUP BY city_ascii
--HAVING COUNT(DISTINCT iso3) > 1;